# Phase III Part B: LightGBM Ranking Strategy
Trains a LightGBM lambdarank model on the feature table using walk-forward validation.
At each month end, ranks the 5 ETFs and allocates using risk parity weighting.

Walk-forward windows: 3 years train â†’ 1 year test â†’ roll 1 year forward.

In [1]:
import ast
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

ETFS       = ['SPY', 'EFA', 'IEF', 'VNQ', 'DBC']
MACRO_COLS = ['FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO']
TRAIN_MONTHS = 36  # 3 years
TEST_MONTHS  = 12  # 1 year

## Step 1: Load and Parse Feature Table

In [2]:
raw = pd.read_csv('../feature_engineer_csv.csv', parse_dates=['date'], index_col='date')
print(f'Loaded feature table: {raw.shape}')

# Parse bollinger_bands dict strings into separate upper/middle/lower columns
def parse_bollinger(df, ticker):
    col = f'{ticker}_bollinger_bands'
    parsed = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else {})
    df[f'{ticker}_bb_upper']  = parsed.apply(lambda d: d.get('upper', np.nan))
    df[f'{ticker}_bb_middle'] = parsed.apply(lambda d: d.get('middle', np.nan))
    df[f'{ticker}_bb_lower']  = parsed.apply(lambda d: d.get('lower', np.nan))
    df = df.drop(columns=[col])
    return df

features = raw.copy()
for etf in ETFS:
    features = parse_bollinger(features, etf)

# Drop spoofed OHLCV columns (high/low/open/volume are identical to close)
drop_cols = [f'{etf}_{c}' for etf in ETFS for c in ['high', 'low', 'open', 'volume']]
features = features.drop(columns=drop_cols)

print(f'Feature table after parsing: {features.shape}')
print(f'Columns: {features.columns.tolist()}')

Loaded feature table: (207, 65)
Feature table after parsing: (207, 55)
Columns: ['SPY_close', 'SPY_macd', 'SPY_rsi', 'SPY_sma', 'SPY_dist_from_sma', 'SPY_dist_from_10m_sma', 'SPY_above_10m_sma', 'EFA_close', 'EFA_macd', 'EFA_rsi', 'EFA_sma', 'EFA_dist_from_sma', 'EFA_dist_from_10m_sma', 'EFA_above_10m_sma', 'IEF_close', 'IEF_macd', 'IEF_rsi', 'IEF_sma', 'IEF_dist_from_sma', 'IEF_dist_from_10m_sma', 'IEF_above_10m_sma', 'VNQ_close', 'VNQ_macd', 'VNQ_rsi', 'VNQ_sma', 'VNQ_dist_from_sma', 'VNQ_dist_from_10m_sma', 'VNQ_above_10m_sma', 'DBC_close', 'DBC_macd', 'DBC_rsi', 'DBC_sma', 'DBC_dist_from_sma', 'DBC_dist_from_10m_sma', 'DBC_above_10m_sma', 'FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO', 'TB3MS', 'SPY_bb_upper', 'SPY_bb_middle', 'SPY_bb_lower', 'EFA_bb_upper', 'EFA_bb_middle', 'EFA_bb_lower', 'IEF_bb_upper', 'IEF_bb_middle', 'IEF_bb_lower', 'VNQ_bb_upper', 'VNQ_bb_middle', 'VNQ_bb_lower', 'DBC_bb_upper', 'DBC_bb_middle', 'DBC_bb_lower']


## Step 2: Convert to Long Format and Add Labels
Each row = one ETF at one month end.
Label = rank of ETF by actual next-month return (4 = best, 0 = worst).

In [3]:
# Load prices to compute actual next-month returns (for labelling only)
prices = pd.read_csv('../cleaned_prices.csv', index_col=0, parse_dates=True)
next_month_returns = prices[ETFS].pct_change().shift(-1)  # return EARNED next month

# Build long-format feature table
per_etf_feature_cols = ['close', 'rsi', 'macd', 'sma', 'dist_from_sma',
                         'bb_upper', 'bb_middle', 'bb_lower',
                         'dist_from_10m_sma', 'above_10m_sma']

rows = []
for date in features.index:
    if date not in next_month_returns.index:
        continue
    macro_vals = features.loc[date, MACRO_COLS].to_dict()
    next_rets   = next_month_returns.loc[date]

    for etf in ETFS:
        etf_feats = {col: features.loc[date, f'{etf}_{col}']
                     for col in per_etf_feature_cols
                     if f'{etf}_{col}' in features.columns}
        row = {'date': date, 'ticker': etf, 'next_return': next_rets[etf]}
        row.update(etf_feats)
        row.update(macro_vals)
        rows.append(row)

long_df = pd.DataFrame(rows).set_index('date').sort_index()

# Rank ETFs within each month: 4 = best next-month return, 0 = worst
long_df['label'] = long_df.groupby('date')['next_return'].rank(method='first').astype(int) - 1

# Drop NaN rows (last month has no next_return)
long_df = long_df.dropna(subset=['next_return'])

print(f'Long-format table: {long_df.shape}')
print(f'Date range: {long_df.index.min().date()} to {long_df.index.max().date()}')
print(f'Columns: {long_df.columns.tolist()}')
long_df.head(10)

Long-format table: (1035, 17)
Date range: 2007-09-30 to 2024-11-30
Columns: ['ticker', 'next_return', 'close', 'rsi', 'macd', 'sma', 'dist_from_sma', 'bb_upper', 'bb_middle', 'bb_lower', 'dist_from_10m_sma', 'above_10m_sma', 'FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO', 'label']


,ticker,next_return,close,rsi,macd,sma,dist_from_sma,bb_upper,bb_middle,bb_lower,dist_from_10m_sma,above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,label
date,,,,,,,,,,,,,,,,,
2007-09-30,SPY,0.013566,108.383347,75.091531,NaN,97.266314,0.114295,111.091657,97.266314,83.440970,0.048917,1,4.94,208.292,0.62,114.3570,1
2007-09-30,EFA,0.042499,46.893772,82.834205,NaN,40.742860,0.150969,48.426668,40.742860,33.059053,0.060478,1,4.94,208.292,0.62,114.3570,3
2007-09-30,IEF,0.010770,52.976494,72.067278,NaN,50.256051,0.054132,53.340063,50.256051,47.172039,0.030467,1,4.94,208.292,0.62,114.3570,0
2007-09-30,VNQ,0.020990,32.989197,59.080698,NaN,32.316261,0.020823,38.465244,32.316261,26.167277,-0.037595,0,4.94,208.292,0.62,114.3570,2
2007-09-30,DBC,0.085023,22.830555,72.026633,NaN,20.241360,0.127916,22.121984,20.241360,18.360735,0.098347,1,4.94,208.292,0.62,114.3570,4
2007-10-31,SPY,-0.038732,109.853699,76.353697,NaN,98.345081,0.117023,112.558347,98.345081,84.131816,0.052398,1,4.76,208.903,0.54,113.9957,1
2007-10-31,EFA,-0.036237,48.886711,85.172477,NaN,41.451774,0.179364,49.389799,41.451774,33.513750,0.087561,1,4.76,208.903,0.54,113.9957,2
2007-10-31,IEF,0.040475,53.547043,74.209892,NaN,50.490171,0.060544,53.817452,50.490171,47.162890,0.035431,1,4.76,208.903,0.54,113.9957,4
2007-10-31,VNQ,-0.094709,33.681652,60.603193,NaN,32.601696,0.033126,38.439854,32.601696,26.763538,-0.014908,0,4.76,208.903,0.54,113.9957,0


## Step 3: Walk-Forward Validation + LightGBM Training

In [4]:
feature_cols = [c for c in long_df.columns if c not in ['ticker', 'next_return', 'label']]

dates      = sorted(long_df.index.unique())
n_dates    = len(dates)
all_preds  = []

i = 0
while i + TRAIN_MONTHS + TEST_MONTHS <= n_dates:
    train_dates = dates[i : i + TRAIN_MONTHS]
    test_dates  = dates[i + TRAIN_MONTHS : i + TRAIN_MONTHS + TEST_MONTHS]

    # Drop rows where MACD is NaN rather than filling with 0
    # MACD needs ~26 months of history; early rows are genuinely missing, not zero
    train_df = long_df[long_df.index.isin(train_dates)].dropna(subset=['macd'])
    test_df  = long_df[long_df.index.isin(test_dates)]

    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df['label']
    groups  = train_df.groupby('date').size().values  # recomputed after dropna

    X_test  = test_df[feature_cols].fillna(0)

    train_data = lgb.Dataset(X_train, label=y_train, group=groups)

    params = {
        'objective':    'lambdarank',
        'metric':       'ndcg',
        'learning_rate': 0.05,
        'num_leaves':   31,
        'verbose':      -1,
    }

    model = lgb.train(params, train_data, num_boost_round=100)
    scores = model.predict(X_test)

    for j, (idx, row) in enumerate(test_df.iterrows()):
        all_preds.append({'date': idx, 'ticker': row['ticker'], 'score': scores[j]})

    i += TEST_MONTHS

preds_df = pd.DataFrame(all_preds)
print(f'Walk-forward complete: {len(preds_df)} predictions across {preds_df["date"].nunique()} months.')

Walk-forward complete: 840 predictions across 168 months.


## Step 4: Risk Parity Weighting and Backtest

In [5]:
# 60-day daily volatility for risk parity weights
prices_daily = pd.read_csv("../cleaned_prices_daily.csv", index_col=0, parse_dates=True)
vol_60d = prices_daily[ETFS].pct_change().rolling(60).std()

monthly_returns = prices[ETFS].pct_change()

# Map each signal date to the following month's return date
mr_dates = sorted(monthly_returns.index)
next_date_map = {mr_dates[i]: mr_dates[i + 1] for i in range(len(mr_dates) - 1)}

TOP_N = 3     # ETFs selected by ML ranking each month
COST  = 0.001 # 0.1% per unit of one-way turnover

prev_weights = pd.Series(0.0, index=ETFS)
portfolio_returns = []

for date, group in preds_df.groupby("date"):
    ret_date = next_date_map.get(date)
    if ret_date is None or ret_date not in monthly_returns.index:
        continue

    # Select top-N ETFs by LightGBM score
    top_etfs = group.nlargest(TOP_N, 'score')['ticker'].tolist()

    # Apply risk parity among the top-N only
    vols = vol_60d.asof(date)[top_etfs].replace(0, np.nan).dropna()
    if vols.empty:
        continue

    inv_vol = 1.0 / vols
    weights = inv_vol / inv_vol.sum()

    full_weights = weights.reindex(ETFS, fill_value=0.0)
    turnover = (full_weights - prev_weights).abs().sum() / 2
    txn_cost = COST * turnover

    ret = (monthly_returns.loc[ret_date, weights.index] * weights).sum() - txn_cost
    portfolio_returns.append({"date": ret_date, "return": ret, "n_invested": len(weights)})
    prev_weights = full_weights

ml_returns = pd.DataFrame(portfolio_returns).set_index("date")
print(f"ML backtest complete: {len(ml_returns)} months.")
print(f"Date range: {ml_returns.index.min().date()} to {ml_returns.index.max().date()}")
ml_returns.head(10)

ML backtest complete: 168 months.
Date range: 2010-10-31 to 2024-09-30


,return,n_invested
date,,
2010-10-31,0.016423,3
2010-11-30,-0.014743,3
2010-12-31,0.019863,3
2011-01-31,0.012834,3
2011-02-28,0.019587,3
2011-03-31,-0.006478,3
2011-04-30,0.028695,3
2011-05-31,0.006170,3
2011-06-30,-0.009393,3


## Step 5: Save Results

In [6]:
ml_returns.to_csv('../ml_returns.csv')
print('Saved: ml_returns.csv')

Saved: ml_returns.csv
